# Hilbert-Geometry Diagnostics
## Version 2: CICIoT2023 | Day-23 final MAQT checkpoint

## Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pennylane as qp
from pennylane import numpy as np
import torch

import pandas as pd

import json
from pathlib import Path

In [2]:
from scripts.constants import DEFAULT_BATCH_SIZE, DEFAULT_NOISE_RATE, DEFAULT_SEED
from scripts.data import load_split, to_angles
from scripts.circuit import build_forward_circuit, create_quantum_device
from scripts.utils import get_torch_device
from scripts.hilbert import hilbert_geometry_diagnostics, print_h1_report

In [3]:
print(f"PyTorch version: {torch.__version__}")
print(f"PennyLane version: {qp.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.12.1+cu126
PennyLane version: 0.45.1
CUDA available: True


## Config

In [4]:
NOTEBOOK_NAME = "11.hilbert-geometry-diagnostics_v2-ciciot2023"
DAY23_NOTEBOOK = "final-ciciot2023-maqt-train"

dataset = "CICIoT2023"
target_col = "label_multiclass"
data_path = f"data/FROZEN/{dataset}"
frozen_feat_path = "team-artifacts/teamC_week3_FROZEN_handoff.json"

SEED = DEFAULT_SEED
MAX_PER_CLASS = 50  # H1 subsample per class (None = all)

## Load Team C's FROZEN Features

In [5]:
handoff = json.loads(Path(frozen_feat_path).read_text())
FEATURE_COLS = list(handoff["frozen_subsets"][dataset]["features"])
print(f"Team C selector: {handoff['frozen_subsets'][dataset]['selector']}")
print(f"k={len(FEATURE_COLS)} features: {FEATURE_COLS}")

Team C selector: MI
k=10 features: ['duration', 'n_pkts_total', 'n_bytes_total', 'rate', 'pkt_size_min', 'pkt_size_max', 'pkt_size_mean', 'iat', 'protocol', 'conn_state']


## Load Team A's FROZEN Train Split

Only the train split is needed for H1 fidelity-gap measurement against Day-23 prototypes (no EDA / balancing / retrain).

In [6]:
X_train, y_train, class_names, df_train = load_split(
    data_path, "train", target_col, categories=None, csv=True,
    selected_cols=FEATURE_COLS, return_df=True,
)

print(f"train: {X_train.shape}, y={y_train.shape}")
print(f"classes: {class_names}")
df_train.head(3)

train: (151049, 10), y=(151049,)
classes: ['Backdoor_Malware', 'BenignTraffic', 'BrowserHijacking', 'CommandInjection', 'DDoS-ACK_Fragmentation', 'DDoS-HTTP_Flood', 'DDoS-ICMP_Flood', 'DDoS-ICMP_Fragmentation', 'DDoS-PSHACK_Flood', 'DDoS-RSTFINFlood', 'DDoS-SYN_Flood', 'DDoS-SlowLoris', 'DDoS-SynonymousIP_Flood', 'DDoS-TCP_Flood', 'DDoS-UDP_Flood', 'DDoS-UDP_Fragmentation', 'DNS_Spoofing', 'DictionaryBruteForce', 'DoS-HTTP_Flood', 'DoS-SYN_Flood', 'DoS-TCP_Flood', 'DoS-UDP_Flood', 'MITM-ArpSpoofing', 'Recon-HostDiscovery', 'Recon-OSScan', 'Recon-PingSweep', 'Recon-PortScan', 'SqlInjection', 'Uploading_Attack', 'VulnerabilityScan', 'XSS']


,duration,n_pkts_total,n_bytes_total,rate,pkt_size_min,pkt_size_max,pkt_size_mean,iat,protocol,conn_state,label_multiclass
0,0.114311,0.627152,0.005499,-0.306870,0.000000,0.050420,0.000000,0.855064,0.000000,-1.368171,DDoS-RSTFINFlood
1,11.888079,59.170326,2.426720,0.744359,2.613771,22.605136,11.713125,-314.863555,-1.322219,-1.217438,BenignTraffic
2,0.000000,-0.250922,-3.264916,0.257804,-3.259715,-2.850162,-3.252384,0.020018,-1.000000,0.000000,DDoS-ICMP_Flood


## Load Day-23 FROZEN Checkpoint

Reuse $\theta^\star$, prototypes, and the train-fitted scaler / PCA / angle bounds from Day-23.

In [7]:
LOAD_CHECKPOINT = True
ARTIFACTS_DIR = Path("final_notebooks") / "final_artifacts"
CHECKPOINT_CANDIDATES = [
    ARTIFACTS_DIR / f"{DAY23_NOTEBOOK}-checkpoint.pt",
]

batch_size = DEFAULT_BATCH_SIZE
seed = SEED

In [8]:
ckpt_path = next((p for p in CHECKPOINT_CANDIDATES if p.exists()), None)

if not (LOAD_CHECKPOINT and ckpt_path is not None):
    raise FileNotFoundError(
        f"Day-23 checkpoint not found under {CHECKPOINT_CANDIDATES}. "
        f"Publish it from final_notebooks/{DAY23_NOTEBOOK}.ipynb first."
    )

ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

theta_star = ckpt["theta"]
if not isinstance(theta_star, torch.nn.Parameter):
    theta_star = torch.nn.Parameter(theta_star)

prototypes = {int(k): v for k, v in ckpt["prototypes"].items()}
class_names = list(ckpt.get("class_names", class_names))
num_classes = int(ckpt.get("num_classes", len(class_names)))
num_qubits = int(ckpt["num_qubits"])
num_layers = int(ckpt.get("num_layers", 2))
noise_rate = float(ckpt.get("noise_rate", DEFAULT_NOISE_RATE))

scaler = ckpt["scaler"]
pca = ckpt.get("pca")
USE_PCA = bool(ckpt.get("use_pca", pca is not None))
x_min = np.asarray(ckpt["angle_x_min"])
x_max = np.asarray(ckpt["angle_x_max"])
angle_max = float(ckpt.get("angle_max", np.pi))
k95 = int(ckpt.get("k95", num_qubits))

print(f"loaded checkpoint: {ckpt_path}")
print(f"theta shape: {tuple(theta_star.shape)} | prototypes: {len(prototypes)}")
print(f"num_qubits={num_qubits} | num_layers={num_layers} | USE_PCA={USE_PCA} | k95={k95}")
print(f"feature_cols: {len(ckpt.get('feature_cols', FEATURE_COLS))} | classes: {class_names}")

loaded checkpoint: final_notebooks/final_artifacts/final-ciciot2023-maqt-train-checkpoint.pt
theta shape: (3, 6, 3) | prototypes: 31
num_qubits=6 | num_layers=3 | USE_PCA=True | k95=6
feature_cols: 10 | classes: ['Backdoor_Malware', 'BenignTraffic', 'BrowserHijacking', 'CommandInjection', 'DDoS-ACK_Fragmentation', 'DDoS-HTTP_Flood', 'DDoS-ICMP_Flood', 'DDoS-ICMP_Fragmentation', 'DDoS-PSHACK_Flood', 'DDoS-RSTFINFlood', 'DDoS-SYN_Flood', 'DDoS-SlowLoris', 'DDoS-SynonymousIP_Flood', 'DDoS-TCP_Flood', 'DDoS-UDP_Flood', 'DDoS-UDP_Fragmentation', 'DNS_Spoofing', 'DictionaryBruteForce', 'DoS-HTTP_Flood', 'DoS-SYN_Flood', 'DoS-TCP_Flood', 'DoS-UDP_Flood', 'MITM-ArpSpoofing', 'Recon-HostDiscovery', 'Recon-OSScan', 'Recon-PingSweep', 'Recon-PortScan', 'SqlInjection', 'Uploading_Attack', 'VulnerabilityScan', 'XSS']


## Mapping Features $\rightarrow$ [0, $\pi$]

In [9]:
# raw -> scale -> optional pca -> [0, pi] using Day-23 train bounds
pca_for_angles = pca if USE_PCA else None
X_train_angles = to_angles(X_train, scaler, x_min, x_max, pca=pca_for_angles, angle_max=angle_max)

print(f"angles train: {X_train_angles.shape} | USE_PCA={USE_PCA}")

angles train: (151049, 6) | USE_PCA=True


## Quantum Circuit: Angle Encoding, Data Reuploading, and Adding Noise

$x\xrightarrow[\text{encode}]{\phi(x)}\ket{\psi(x)}\xrightarrow[\text{variational}]{U(\theta)}\ket{\Phi(x)}\xrightarrow{\Lambda_p}\rho(x)$
- $x$: classical data
- $\phi(x)$: `qp.AngleEmbedding()`
- $\ket{\psi(x)}$: quantum state after encoding
- $U(\theta)$: `qp.StronglyEntanglingLayers()`
- $\ket{\Phi(x)}$: quantum state after variational transform
- $\rho(x)=\Lambda_p(\ket{\Phi(x)}\bra{\Phi(x)})$: standard depolarization channel to model NISQ noise

In [10]:
print(f"num_qubits={num_qubits} (from Day-23) | num_layers={num_layers} | noise_rate={noise_rate}")

# initialize devices
device = get_torch_device()
dev = create_quantum_device(num_qubits)

# define circuit
forward_circuit = build_forward_circuit(dev, num_qubits, num_layers, noise_rate=noise_rate)

# move model tensors to device
theta_star = theta_star.to(device)
if not isinstance(theta_star, torch.nn.Parameter):
    theta_star = torch.nn.Parameter(theta_star)
prototypes = {int(k): v.to(device) for k, v in prototypes.items()}

print(f"device={device}")

num_qubits=6 (from Day-23) | num_layers=3 | noise_rate=0.01
device=cuda


## H1 Diagnostics

### Loss Functions
- **Intra-class loss** (infidelity to own prototype):
  $$
  L_{\mathrm{intra}}
  = \frac{1}{|\mathcal{C}|}
    \sum_{c \in \mathcal{C}}
    \mathbb{E}_{x \sim c}
    \big[1 - F(\rho(x), \rho_c)\big]
  $$
- **Inter-class loss** (negative mean prototype separation):
  $$
  L_{\mathrm{inter}}
  = - \frac{1}{|\mathcal{P}|}
    \sum_{(c,c') \in \mathcal{P}}
    D_{\mathrm{tr}}(\rho_c, \rho_{c'})
  $$
  where $\mathcal{P}$ is the set of unordered class pairs and $D_{\mathrm{tr}}$ is trace distance.

### Fidelity Gap
$$
\Delta F = \underbrace{\overline{F}_{\mathrm{intra}}}_{\mathrm{mean}_{c}\,(\mathrm{mean}_{x \in c} F(\rho(x), \rho_c))} - \underbrace{\overline{F}_{\mathrm{inter}}}_{\mathrm{mean}_{(c,c')} F(\rho_c,\rho_{c'})}
$$

where:

- $\overline{F}_{\mathrm{intra}} \approx 1 - L_{\mathrm{intra}}$ (proxy from the loss)
- $\overline{F}_{\mathrm{intra}} = \mathrm{mean\_intra\_fid} = \mathrm{mean}_{c}\,(\mathrm{mean}_{x \in c} F(\rho(x), \rho_c))$ (direct definition / H1 measurement)
    - $\mathrm{mean\_intra\_fid}_c = \mathrm{mean}_{x \in c} F(\rho(x), \rho_c)$ (per-class intra fidelity)
- $\overline{F}_{\mathrm{inter}} = \mathrm{mean\_inter\_fid} = \mathrm{mean}_{(c,c')} F(\rho_c,\rho_{c'})$
  (explicit fidelity; not from $L_{\mathrm{inter}}$)

### Ideal Trends
| Goal | Geometry | $L$ | Fidelity |
|---|---|---|---|
| Same class tighter | closer to $\rho_c$ | $L_{\mathrm{intra}} \downarrow$ | $F(\rho(x),\rho_c) \uparrow$ |
| Different classes farther | prototypes separate | $L_{\mathrm{inter}} \downarrow$ (more negative; $D_{\mathrm{tr}} \uparrow$) | $F(\rho_c,\rho_{c'}) \downarrow$ |
| Better Hilbert margin | - | - | $\Delta F \uparrow$ |

In [11]:
h1 = hilbert_geometry_diagnostics(
    theta_star, X_train_angles, y_train, prototypes, forward_circuit,
    class_names=class_names, device=device, max_per_class=MAX_PER_CLASS,
    seed=seed, batch_size=batch_size,
)
print_h1_report(h1)

=== H1 Hilbert geometry (fidelity gaps) ===
mean intra-class fidelity : 0.7816
mean inter-class fidelity : 0.6233
fidelity gap (intra-inter): 0.1583  ← want ↑
mean inter trace distance : 0.6717  ← want ↑

per-class intra fidelity:
  Backdoor_Malware             n=  50  F=0.6421
  BenignTraffic                n=  50  F=0.6870
  BrowserHijacking             n=  50  F=0.6047
  CommandInjection             n=  50  F=0.5964
  DDoS-ACK_Fragmentation       n=  50  F=0.8974
  DDoS-HTTP_Flood              n=  50  F=0.8435
  DDoS-ICMP_Flood              n=  50  F=0.9666
  DDoS-ICMP_Fragmentation      n=  50  F=0.9362
  DDoS-PSHACK_Flood            n=  50  F=0.9379
  DDoS-RSTFINFlood             n=  50  F=0.9409
  DDoS-SYN_Flood               n=  50  F=0.9299
  DDoS-SlowLoris               n=  50  F=0.7924
  DDoS-SynonymousIP_Flood      n=  50  F=0.9517
  DDoS-TCP_Flood               n=  50  F=0.9337
  DDoS-UDP_Flood               n=  50  F=0.9951
  DDoS-UDP_Fragmentation       n=  50  F=0.9586
 

In [12]:
# within-class (intra)
display(pd.DataFrame(h1["per_class"]).T)
display(pd.DataFrame(h1["pairs"]).sort_values("pair_inter_fid", ascending=False))

,n,mean_intra_fid_c,mean_intra_infidelity_c
Backdoor_Malware,50.0,0.642103,0.357897
BenignTraffic,50.0,0.687049,0.312951
BrowserHijacking,50.0,0.604657,0.395343
CommandInjection,50.0,0.596398,0.403602
DDoS-ACK_Fragmentation,50.0,0.897441,0.102559
DDoS-HTTP_Flood,50.0,0.843543,0.156457
DDoS-ICMP_Flood,50.0,0.966577,0.033423
DDoS-ICMP_Fragmentation,50.0,0.936198,0.063802
DDoS-PSHACK_Flood,50.0,0.937931,0.062069
DDoS-RSTFINFlood,50.0,0.940947,0.059053


,pair,pair_inter_fid,pair_trace_distance
212,DDoS-PSHACK_Flood↔DDoS-RSTFINFlood,0.996325,0.080941
300,DDoS-SynonymousIP_Flood↔DoS-SYN_Flood,0.995361,0.069407
445,Recon-OSScan↔Recon-PortScan,0.989083,0.124859
256,DDoS-SYN_Flood↔DDoS-SynonymousIP_Flood,0.985752,0.155090
213,DDoS-PSHACK_Flood↔DDoS-SYN_Flood,0.984414,0.170175
...,...,...,...
8,Backdoor_Malware↔DDoS-RSTFINFlood,0.377282,0.870692
218,DDoS-PSHACK_Flood↔DDoS-UDP_Fragmentation,0.356933,0.883895
189,DDoS-ICMP_Fragmentation↔DDoS-PSHACK_Flood,0.354565,0.879838
239,DDoS-RSTFINFlood↔DDoS-UDP_Fragmentation,0.349656,0.885950


## Logging

In [13]:
out_dir = Path("logs")
out_dir.mkdir(parents=True, exist_ok=True)
log_path = out_dir / f"{NOTEBOOK_NAME}.log"

lines = [
    f"notebook={NOTEBOOK_NAME}",
    f"checkpoint={ckpt_path}",
    f"dataset={dataset}",
    f"max_per_class={MAX_PER_CLASS}",
    f"mean_intra_fid={h1['mean_intra_fid']:.6f}",
    f"mean_inter_fid={h1['mean_inter_fid']:.6f}",
    f"fidelity_gap={h1['fidelity_gap']:.6f}",
    f"mean_inter_trace_distance={h1['mean_inter_trace_distance']:.6f}",
]
for name, row in h1["per_class"].items():
    lines.append(f"intra[{name}]: n={row['n']} F={row['mean_intra_fid_c']:.6f}")
for row in h1["pairs"]:
    lines.append(
        f"pair[{row['pair']}]: F={row['pair_inter_fid']:.6f} TD={row['pair_trace_distance']:.6f}"
    )

log_path.write_text("\n".join(lines) + "\n")
print(f"wrote {log_path}")

wrote logs/11.hilbert-geometry-diagnostics_v2-ciciot2023.log
